In [105]:
from torch.utils import data
from torch.utils.data import DataLoader,Dataset
import pandas as pd
import torch.nn as nn
import numpy as np
import torch

covid_data= pd.read_csv('covid.train.csv')
covid_data.head()


,id,AL,AK,AZ,AR,CA,CO,CT,FL,GA,...,restaurant.2,spent_time.2,large_event.2,public_transit.2,anxious.2,depressed.2,felt_isolated.2,worried_become_ill.2,worried_finances.2,tested_positive.2
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.812411,43.430423,16.151527,1.602635,15.409449,12.088688,16.702086,53.991549,43.604229,20.704935
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.682974,43.196313,16.123386,1.641863,15.230063,11.809047,16.506973,54.185521,42.665766,21.292911
2,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,23.593983,43.362200,16.159971,1.677523,15.717207,12.355918,16.273294,53.637069,42.972417,21.166656
3,3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.576992,42.954574,15.544373,1.578030,15.295650,12.218123,16.045504,52.446223,42.907472,19.896607
4,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,22.091433,43.290957,15.214655,1.641667,14.778802,12.417256,16.134238,52.560315,43.321985,20.178428


In [106]:

# 1.定义数据集 
class COVID19Dataset(Dataset):
    def __init__(self,file_path,mode='train',target_only=False) -> None:
        super().__init__()
        self.mode = mode

        data = pd.read_csv(file_path)
        data = np.array(data[1:])[:,1:].astype(float)

        if not target_only:
            feats = list(range(93))
        else:
            feats = list(range(40))+[57,75]


        if mode == 'test':
            data = data[:,feats]
            self.data = torch.FloatTensor(data)
        else:
            target = data[:,-1]
            data = data[:,feats]
            if mode == 'train':
                feather = [i for i in range(len(data)) if i % 10 != 0]
            elif mode == 'dev':
                feather = [i for i in range(len(data)) if i % 10 == 0]

            self.data = torch.FloatTensor(data[feather])
            self.target = torch.FloatTensor(target[feather])
        
        self.dim = self.data.shape[1]

        print('Finished reading the {} set of COVID19 Dataset ({} samples found, each dim = {})'
              .format(mode,len(self.data),self.dim))
        
    def __getitem__(self,index):
        if self.mode in ['train','dev']:
            return self.data[index],self.target[index]
        else:
            return self.data[index]
    
    def __len__(self):
        return len(self.data)
            


    


In [107]:
def get_dataloader(path,mode,batch_size,n_jobs=0,target_only=False):
    dataset = COVID19Dataset(path,mode=mode,target_only=target_only)
    dataloader = DataLoader(
        dataset,batch_size,
        shuffle=(mode == 'train'),drop_last=False,
        num_workers=n_jobs,pin_memory=True
    )
    return dataloader

In [108]:
# 2.定义模型
class NeuralNet(nn.Module):
    def __init__(self,input_dim):
        super(NeuralNet,self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )


    def forward(self,x):
        return self.net(x).squeeze(1)
    


In [109]:
# 3.定义损失函数
def criterion(pred,target):
    return nn.MSELoss(reduction='mean')
    


In [110]:

# 4.定义优化器
def optimizer(model,lr,weight_decay):
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


In [111]:
# 5.定义设备
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    

In [112]:
config = {
    'max_epochs': 3000,
    'batch_size': 128,
    'optimizer': 'SGD',
    'optim_hparas': {
        'lr': 0.01,
        'weight_decay': 1e-4
    },
    'device': get_device(),
    'save_path': 'models/model.pth',
    'early_stop': 200,
    'file_path': 'covid.train.csv'
    }

In [113]:
# 6.定义评估函数
def dev(dv_set,model,device):
    model.eval()
    total_loss = 0
    for x,y in dv_set:
        x,y = x.to(device),y.to(device)
        with torch.no_grad():
            pred = model(x)
            mse_loss = criterion(pred,y)
        total_loss += mse_loss.detach().cpu().item() * len(x)
    total_loss = total_loss / len(dv_set.dataset)
    return total_loss



In [ ]:
# 7.定义训练函数
def train(tr_set,dv_set,model,config,device):
    max_epochs = config['max_epochs']
    batch_size = config['batch_size']
    lr = config['optim_hparas']['lr']
    weight_decay = config['optim_hparas']['weight_decay']


    optimizer = optimizer(model,lr,weight_decay)

    min_mse = 1000
    loss_record = {'train':[],'dev':[]}
    early_stop_cnt = 0
    epoch = 0

    while epoch < max_epochs:
        model.train()
        for x,y in tr_set:
            x,y = x.to(device),y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred,y)
            loss.backward()
            optimizer.step()
            loss_record['train'].append(loss.detach().cpu().item())

        dev_mse = dev(dv_set,model,device)
        if dev_mse < min_mse:
            min_mse = dev_mse
            print('Saving model (epoch = {:4d}, loss = {:.4f})'
                  .format(epoch+1,min_mse))
            torch.save(model.state_dict(),config['save_path'])
            early_stop_cnt = 0
        else:
            early_stop_cnt += 1
            
        loss_record['dev'].append(dev_mse)
        epoch += 1
        if early_stop_cnt > config['early_stop']:
            print('Early stopping at epoch {:4d}...'.format(epoch))
            break
    
    return min_mse,loss_record

    
        

In [115]:
# 8.定义测试函数
def test(tt_set,model,device):
    model.eval()
    preds = []
    for x in tt_set:
        x = x.to(device)
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu())
    preds = torch.cat(preds,dim=0).numpy()
    return preds

In [116]:

tr_set = get_dataloader(config['file_path'], 'train', config['batch_size'])
dv_set = get_dataloader(config['file_path'], 'dev', config['batch_size'])
tt_set = get_dataloader(config['file_path'], 'test', config['batch_size'])




Finished reading the train set of COVID19 Dataset (2429 samples found, each dim = 93)
Finished reading the dev set of COVID19 Dataset (270 samples found, each dim = 93)
Finished reading the test set of COVID19 Dataset (2699 samples found, each dim = 93)


In [117]:
model = NeuralNet(tr_set.dataset.dim).to(config['device'])


In [118]:
model_loss, model_loss_record = train(tr_set, dv_set, model, config, config['device'])

UnboundLocalError: cannot access local variable 'optimizer' where it is not associated with a value